# The Great British Bake Off: Exploratory Data Analysis & Statistics 🎂

Welcome to the exploratory analysis starter notebook for **The Great British Bake Off (GBBO)** dataset (broadcast as *The Great British Baking Show* in North America).

This notebook demonstrates how to load the processed relational tables in Python, analyze historical baking trends, evaluate the impact of **Hollywood Handshakes**, examine demographic patterns, and trace predictive indicators for weekly outcomes.


## 1. Setup & Environment

Import core libraries and configure data directories. We can load from **Apache Parquet**, **CSV**, **Arrow IPC**, or query the unified **SQLite** database (`gbbo.db`).


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import pyarrow.feather as feather

# Resolve data path (supports running from repo root or notebooks/ dir)
DATA_DIR = Path("../data/processed") if Path("../data/processed").exists() else Path("data/processed")
print(f"Using processed data directory: {DATA_DIR.resolve()}")


## 2. Load Processed Datasets

Load the master tidy relational tables (`seasons`, `bakers`, `episodes`, `performances`).


In [ ]:
def load_table(name: str) -> pd.DataFrame:
    parquet_file = DATA_DIR / f"{name}.parquet"
    if parquet_file.exists():
        return pd.read_parquet(parquet_file)
    return pd.read_csv(DATA_DIR / f"{name}.csv")

seasons = load_table("seasons")
bakers = load_table("bakers")
episodes = load_table("episodes")
performances = load_table("performances")

print(f"Loaded {len(seasons)} seasons, {len(bakers)} bakers, {len(episodes)} episodes, {len(performances)} performances.")


## 3. High-Level Series Overview & Historical Champions

Inspect the champions across all broadcast series alongside network broadcasters and filming locations.


In [ ]:
champions = seasons[["series_number", "year", "winner", "network", "location"]].dropna(subset=["winner"])
champions.sort_values("series_number")


## 4. The Hollywood Handshake: Frequency & Impact

The **Hollywood Handshake** is one of the highest honors Paul Hollywood awards to exceptional Signature or Showstopper bakes. How common are they across series, and who received the most?


In [ ]:
# Extract series sequence from episode_id
performances["series_number"] = performances["episode_id"].str.extract(r"s(\d+)e").astype(int)

# Handshakes per series
hs_per_series = performances[performances["hollywood_handshake"] == True].groupby("series_number").size().reset_index(name="handshake_count")
print("--- Handshakes Awarded Per Series ---")
print(hs_per_series.to_string(index=False))

# Top bakers by cumulative Hollywood Handshakes
top_hs_bakers = (
    performances[performances["hollywood_handshake"] == True]
    .groupby("baker_id")
    .size()
    .reset_index(name="handshakes")
    .merge(bakers[["baker_id", "name", "series_number", "overall_placement"]], on="baker_id")
    .sort_values("handshakes", ascending=False)
)
top_hs_bakers.head(10)


## 5. Technical Challenge Rank vs. Star Baker Correlation

How strongly does finishing first in the blind Technical Challenge correlate with winning the episode's **Star Baker** title?


In [ ]:
# Filter episodes with technical rank recorded
ranked_perfs = performances[performances["technical_rank"].notna()].copy()

# How often did technical winner (rank 1) get Star Baker?
tech_winners = ranked_perfs[ranked_perfs["technical_rank"] == 1]
tech_sb_conversion = (tech_winners["result"] == "STAR_BAKER").mean()

print(f"Total episodes where baker won Technical Challenge: {len(tech_winners)}")
print(f"Conversion to Star Baker: {tech_sb_conversion:.1%} of technical winners were named Star Baker")


## 6. Baker Demographics: Age & Placement

Let's examine the age distribution of series champions versus early eliminations.


In [ ]:
winners = bakers[bakers["overall_placement"] == 1]["age"].dropna()
finalists = bakers[bakers["overall_placement"].isin([1, 2, 3])]["age"].dropna()
all_bakers_age = bakers["age"].dropna()

print(f"Overall Average Age: {all_bakers_age.mean():.1f} ± {all_bakers_age.std():.1f} years")
print(f"Finalists Average Age: {finalists.mean():.1f} ± {finalists.std():.1f} years")
print(f"Champions Average Age: {winners.mean():.1f} ± {winners.std():.1f} years (min: {winners.min():.0f}, max: {winners.max():.0f})")


## 7. Predictive Signals: At-Risk & Star Baker Momentum

In our empirical predictive engine (`gbbo_dataset.predictor`), callouts by the judges (`did_well` and `at_risk`) serve as vital leading indicators.


In [ ]:
# Analyze eliminations across Series 1-16
h_perfs = performances[performances["series_number"] <= 16].copy()
h_perfs = h_perfs.sort_values(["series_number", "baker_id", "episode_id"])

# Lagged indicators from previous episode
h_perfs["prev_at_risk"] = h_perfs.groupby(["series_number", "baker_id"])["at_risk"].shift(1)
eliminated_rows = h_perfs[h_perfs["result"] == "ELIMINATED"]
elim_after_risk_rate = eliminated_rows["prev_at_risk"].dropna().mean()

print(f"Total recorded eliminations (Series 1-16): {len(eliminated_rows)}")
print(f"Rate of eliminated bakers who were placed at risk in prior episode: {elim_after_risk_rate:.1%}")


## 8. Zero-Dependency SQL Querying via SQLite (`gbbo.db`)

The dataset can be queried directly with pure standard library SQL without needing Pandas or external dependencies.


In [ ]:
db_file = DATA_DIR / "gbbo.db"
if db_file.exists():
    conn = sqlite3.connect(db_file)
    query = """
    SELECT 
        b.name,
        b.series_number,
        b.overall_placement,
        COUNT(*) AS star_bakers,
        SUM(CASE WHEN p.hollywood_handshake = 1 THEN 1 ELSE 0 END) AS handshakes
    FROM performances p
    JOIN bakers b ON p.baker_id = b.baker_id
    WHERE p.result = 'STAR_BAKER'
    GROUP BY b.baker_id
    ORDER BY star_bakers DESC
    LIMIT 8;
    """
    sql_df = pd.read_sql_query(query, conn)
    conn.close()
    print(sql_df)
else:
    print(f"{db_file} not found; run task export:sqlite to generate.")


## 9. Apache Arrow IPC & High-Throughput Analytics

Apache Arrow IPC files (`data/processed/arrow/`) enable zero-copy columnar access across Python, R, and Julia.


In [ ]:
arrow_file = DATA_DIR / "arrow" / "bakers.arrow"
if arrow_file.exists():
    arrow_table = feather.read_table(arrow_file)
    print(f"Arrow Schema:\n{arrow_table.schema}")
    print(f"Total Arrow Record Batches: {len(arrow_table.to_batches())}, Rows: {arrow_table.num_rows}")
else:
    print(f"{arrow_file} not found; run task export:arrow to generate.")
